In [ ]:
import sys
from pathlib import Path

# Add src folder to Python path
project_root = Path.cwd().parent
src_path = project_root / 'src' / 'eeg_synthetic'

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f'✓ Path setup complete: {src_path}')

In [ ]:
import os
from data_loader import BCIAUTLoader, plot_normalized_arrays
from classifiers import EEGNetModel, TrainModel
import torch
import pandas as pd
import numpy as np
from scipy import signal
from torch.utils.data import TensorDataset, ConcatDataset
import matplotlib.pyplot as plt
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, roc_curve, accuracy_score
import torch
import numpy as np
from torch.utils.data import TensorDataset
import torch
import numpy as np
from scipy import linalg
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, roc_curve, accuracy_score

In [ ]:
project_root = os.getcwd()
dataset_path = os.path.join(project_root, 'data') 
loader = BCIAUTLoader(dataset_path)
X_train_sess, y_train_sess, subjects_train_ids, session_train_ids = loader.get_data(subjects=[3], sessions=[3], modes=['Train'])
X_test_sess, y_test_sess, subjects_test_ids, session_test_ids   = loader.get_data(subjects=[3], sessions=[3], modes=['Test'])
plot_normalized_arrays(X_train_sess, y_train_sess, X_test_sess, y_test_sess)

In [ ]:
def load_balanced_synthetic_limited(path_c0, path_c1, max_trials=1200):
    """
    Carica esplicitamente i primi 'max_trials' per classe 0 e classe 1.
    """
    rows_to_read = max_trials * 8

    def get_data_from_csv(path):
        if not os.path.exists(path):
            print(f"ERRORE: File non trovato {path}")
            return None, None
        df = pd.read_csv(path, nrows=rows_to_read)
        time_cols = [col for col in df.columns if col.startswith('Time')]
        X = df[time_cols].values
        y = df['Condition'].values
        return X, y
    print(f"Caricamento primi {max_trials} trial Classe 0...")
    X0_rows, y0_rows = get_data_from_csv(path_c0)
    
    print(f"Caricamento primi {max_trials} trial Classe 1...")
    X1_rows, y1_rows = get_data_from_csv(path_c1)
    X0_trial = torch.tensor(X0_rows).view(-1, 8, 128)
    y0_trial = torch.tensor(y0_rows[::8]) 
    
    X1_trial = torch.tensor(X1_rows).view(-1, 8, 128)
    y1_trial = torch.tensor(y1_rows[::8])

    return X0_trial, y0_trial, X1_trial, y1_trial
file_c0 = 'generated_samples/gen_data_c0_s3_sess3.csv'
file_c1 = 'generated_samples/gen_data_c1_s3_sess3.csv'
X0_syn, y0_syn, X1_syn, y1_syn = load_balanced_synthetic_limited(file_c0, file_c1, max_trials=8600)
X_syn_final = torch.cat([X0_syn, X1_syn], dim=0)
y_syn_final = torch.cat([y0_syn, y1_syn], dim=0)
print(f"Forma finale SINTETICI: X={X_syn_final.shape}, y={y_syn_final.shape}")

In [ ]:
def mne_style_iir_filter(data, fs=128):
    low_cutoff = 0.1
    high_cutoff = 15.0
    order = 4
    b, a = signal.butter(order, [low_cutoff, high_cutoff], btype='bandpass', fs=fs)
    return signal.filtfilt(b, a, data, axis=-1)

def apply_final_alignment(x_tensor):
    """
    Apply Filter and Baseline Correction.
    Input: torch.Tensor [Trials, Channels, Time]
    """
    x_np = x_tensor.detach().cpu().numpy()
    
    # 2. Apply IIR Filter (0.1-15Hz)
    x_filtered = mne_style_iir_filter(x_np, fs=128)
    
    # 3. Apply Baseline Correction (Indexes 0-26 for -0.2s a 0s)
    baseline_period = x_filtered[:, :, :26]
    baseline_mean = np.mean(baseline_period, axis=-1, keepdims=True)
    x_aligned = x_filtered - baseline_mean
    
    # 4. Convert back to Tensor
    return torch.from_numpy(x_aligned.copy()).float()

In [ ]:
#Apply filtering and baseline correction to synthetic data and prepare for similarity metrics
X_syn_ready = apply_final_alignment(X_syn_final)
X_syn_filtered = torch.from_numpy(X_syn_ready.detach().cpu().numpy().copy()).float()
#Z-score trial-wise normalization for synthetic data (using trial-wise mean and std)
mu_syn = X_syn_filtered.mean(dim=(1, 2), keepdim=True)
std_syn = X_syn_filtered.std(dim=(1, 2), keepdim=True)
X_syn_filtered = (X_syn_filtered - mu_syn) / (std_syn)
X_grouped_perm = X_syn_filtered.permute(0, 2, 1) 
syn_win = X_grouped_perm[y_syn_final == 0.0]
syn_loss = X_grouped_perm[y_syn_final == 1.0]
X_train_tensor = torch.tensor(X_train_sess).float()
y_train_tensor = torch.tensor(y_train_sess).float()
X_test_sess_tensor = torch.tensor(X_test_sess).float()
y_test_sess_tensor = torch.tensor(y_test_sess).float()
X_train_tensor = apply_final_alignment(X_train_tensor)
X_test_sess_tensor  = apply_final_alignment(X_test_sess_tensor)
# Z-score trial-wise normalization for real data (using trial-wise mean and std)
mu_ref = X_train_tensor.mean(dim=(1, 2), keepdim=True)
std_ref = X_train_tensor.std(dim=(1, 2), keepdim=True)
X_train_real_z = (X_train_tensor - mu_ref) / (std_ref)
mu_ref_test = X_test_sess_tensor.mean(dim=(1, 2), keepdim=True)
std_ref_test = X_test_sess_tensor.std(dim=(1, 2), keepdim=True)
X_test_real_z = (X_test_sess_tensor - mu_ref_test) / (std_ref_test)
X_train_real_ready = X_train_real_z.permute(0, 2, 1)
X_test_real_ready  = X_test_real_z.permute(0, 2, 1)
print(f"Real Data: Train {X_train_real_ready.shape}, Test {X_test_real_ready.shape}")
real_train_0 = X_train_real_ready[y_train_tensor == 0]
real_test_0  = X_test_real_ready[y_test_sess_tensor == 0]
real_train_1 = X_train_real_ready[y_train_tensor == 1]
real_test_1  = X_test_real_ready[y_test_sess_tensor == 1]

In [ ]:
def prepare_eegnet_dataset(X, label_value):
    """
    Take a tensor [Batch, Time, Channels], transpose it to [Batch, 1, 8, 128]
    and create a TensorDataset with the specified label.
    """
    if isinstance(X, np.ndarray):
        X = torch.from_numpy(X).float()

    if X.shape[1] == 128 and X.shape[2] == 8:
        X = X.transpose(1, 2)

    if X.ndim == 3:
        X = X.unsqueeze(1)

    y = torch.full((X.shape[0],), label_value, dtype=torch.long)
    
    return TensorDataset(X, y)


real_0_ds = prepare_eegnet_dataset(real_train_0, 0)
real_1_ds = prepare_eegnet_dataset(real_train_1, 1)

fake_1_ds = prepare_eegnet_dataset(syn_loss, 1)

test_0_piece = prepare_eegnet_dataset(real_test_0, 0)
test_1_piece = prepare_eegnet_dataset(real_test_1, 1)
test_dataset = ConcatDataset([test_0_piece, test_1_piece])

In [ ]:
def get_eegnet_features(model, x_tensor, device):
    """
    EEGNet feature extraction
    """
    model.eval()
    if x_tensor.ndim == 3: # If is [B, 8, 128]
        x_tensor = x_tensor.unsqueeze(1)
    
    x = x_tensor.to(device).float()
    with torch.no_grad():
        x = model.block1(x)
        x = model.block2(x)
        x = model.block3(x)
        feat = model.flatten(x)
    return feat.cpu().numpy()

def evaluate_advanced_metrics(model, dataset, device):
    model.eval()
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=False)
    
    all_probs, all_preds, all_labels = [], [], []

    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device).float()
            logits = model(x)
            probs = torch.softmax(logits, dim=1)
            
            all_probs.extend(probs[:, 1].cpu().numpy()) 
            all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            all_labels.extend(y.numpy())

    return {
        "acc": accuracy_score(all_labels, all_preds),
        "b_acc": balanced_accuracy_score(all_labels, all_preds),
        "auc": roc_auc_score(all_labels, all_probs),
        "fpr": roc_curve(all_labels, all_probs)[0],
        "tpr": roc_curve(all_labels, all_probs)[1]
    }
def calculate_frechet_distance(features_a, features_b):
    """
    Compute Fréchet Distance
    """
    mu1 = np.mean(features_a, axis=0)
    sigma1 = np.cov(features_a, rowvar=False)
    mu2 = np.mean(features_b, axis=0)
    sigma2 = np.cov(features_b, rowvar=False)

    diff = mu1 - mu2

    covmean = linalg.sqrtm(sigma1.dot(sigma2))
    
    if np.iscomplexobj(covmean):
        covmean = covmean.real

    # Formula FID: ||mu1 - mu2||^2 + Tr(sigma1 + sigma2 - 2*sqrt(sigma1*sigma2))
    return diff.dot(diff) + np.trace(sigma1 + sigma2 - 2 * covmean)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_test_1_tensor = test_1_piece.tensors[0]

# Scenarios
scenarios = [
    {"name": "Solo Reali (Baseline)", "ratio": 0.0},
    {"name": "Aumentato (25% Fake)",  "ratio": 0.25},
    {"name": "Aumentato (50% Fake)",  "ratio": 0.5},
    {"name": "Solo Sintetici (TSTR)", "ratio": 0.99}
]

final_results = {}
trained_models = {}

for sc in scenarios:
    name = sc["name"]
    ratio = sc["ratio"]
    print(f"\n>>> Allenamento Scenario: {name} (Ratio: {ratio})")
    
    model = EEGNetModel(chans=8, classes=2, time_points=128).to(device)
    trainer = TrainModel()
    
    trained_m = trainer.train_model(
        model, 
        real_0_ds, 
        real_1_ds, 
        fake_1_ds=fake_1_ds if ratio > 0 else None,
        learning_rate=0.0001, 
        batch_size=64, 
        epochs=300,
        fake_ratio_in_c1=ratio
    )
    
    m = evaluate_advanced_metrics(trained_m, test_dataset, device)
    
    # Compute the FID
    if ratio == 0.0:
        X_train_c1_scenario = real_1_ds.tensors[0]
    elif ratio == 0.99:
        X_train_c1_scenario = fake_1_ds.tensors[0]
    else:
        X_train_c1_scenario = torch.cat([real_1_ds.tensors[0], fake_1_ds.tensors[0]], dim=0)

    feat_train_scenario = get_eegnet_features(trained_m, X_train_c1_scenario, device)
    feat_test_real      = get_eegnet_features(trained_m, X_test_1_tensor, device)
    
    m["fid"] = calculate_frechet_distance(feat_train_scenario, feat_test_real)
    
    final_results[name] = m
    trained_models[name] = trained_m

# ==============================================================================================================
# TABLE SUMMARY OF RESULTS
# ==============================================================================================================
header = f"{'SCENARIO':<32} | {'ACC':>8} | {'B-ACC':>8} | {'AUC':>8} | {'FID (Tr-Te)':>12}"
sep = "-" * len(header)

print("\n" + "="*len(header))
print(f"{'REPORT COMPARATIVO: IMPATTO DELLA DIFFUSION DATA AUGMENTATION':^85}")
print("="*len(header))
print(header)
print(sep)

for name, res in final_results.items():
    print(f"{name:<32} | {res['acc']:>8.4f} | {res['b_acc']:>8.4f} | {res['auc']:>8.4f} | {res['fid']:>12.4f}")

print("="*len(header))

# ==============================================================================================================
# PLOT DELLE CURVE ROC
# ==============================================================================================================
plt.figure(figsize=(10, 7))
colors = ['#E63946', '#F4A261', '#2A9D8F', '#457B9D'] 

for i, (name, m) in enumerate(final_results.items()):
    plt.plot(m['fpr'], m['tpr'], color=colors[i], lw=2.5, 
             label=f"{name} (AUC = {m['auc']:.3f})")

plt.plot([0, 1], [0, 1], color='#1D3557', linestyle='--', lw=1, alpha=0.5)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1-Spec)', fontsize=11)
plt.ylabel('True Positive Rate (Sens)', fontsize=11)
plt.title('Confronto Curve ROC: Analisi dei vari Ratio di Augmentation', fontsize=13, fontweight='bold')
plt.legend(loc="lower right", fontsize=10)
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()